# ErgoVision AI — Optional Ergonomic Feature Classifier

**CBSE Class XII · Artificial Intelligence (417) · Academic session 2026–27**

This notebook explores whether a supervised model can learn to classify *contributor-labelled posture images* from pose-derived geometric features. It does **not** train the MediaPipe pose model. The app uses MediaPipe's pretrained pose estimator and its transparent rule-based screening independently; this classifier is optional.

> **Important:** this is an educational experiment, not a medical device. The public candidate dataset below has only 273 images labelled `Good` / `Bad`, and the source page does not explain how those labels were assigned. Treat the labels as uncertain and the resulting model as a classroom experiment—not clinically meaningful ground truth.

## ML hypothesis

A classifier might identify patterns correlated with labels in a small set of 2D pose features. It may fail on new people, camera views, clothing, lighting or workstation setups. Performance is only measured after the user supplies the dataset and runs this notebook; no results are pre-filled.

**Feature contract (same names/order as the web app):**
1. `trunkInclinationDeg`
2. `headNeckProxyDeg`
3. `leftElbowAngleDeg`
4. `rightElbowAngleDeg`
5. `leftArmTrunkSeparationDeg`
6. `rightArmTrunkSeparationDeg`

These are projected image-space angles, not fully validated anatomical measurements. Missing values are not silently treated as zero.

## 1. Environment setup

The notebook is designed for Google Colab. The pinned packages form a compatible experiment baseline; Colab's managed runtime can change, so review installation output if the environment has moved on. If installation changes the NumPy version, restart the runtime when Colab prompts you.

In [ ]:
%pip -q install mediapipe==0.10.21 numpy==1.26.4 pandas==2.2.3 scikit-learn==1.6.1 matplotlib==3.10.1 pillow==11.1.0 joblib==1.4.2

import sys, os, json, warnings, platform
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix
)
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Python:", platform.python_version())
print("MediaPipe:", getattr(mp, "__version__", "unknown"))
print("NumPy:", np.__version__, "pandas:", pd.__version__, "scikit-learn:", __import__("sklearn").__version__)

## 2. Dataset source, license and acquisition

### Candidate evaluated

- **Name:** [sitting posture Computer Vision Dataset](https://universe.roboflow.com/ikornproject/sitting-posture-rofqf) by IKORNProject on Roboflow Universe.
- **Source page lists:** 273 images and two labels, `Good` and `Bad`; the task is presented as keypoint detection.
- **License displayed on the source page:** **CC BY 4.0**. Review the current page and attribution terms before use; credit IKORNProject and link the license.
- **Known limitation:** the source page has no substantive dataset description defining the label criteria, subject count, collection procedure, or validation. Therefore it is only a candidate for a small *exploratory* exercise. Do not call its labels clinical ground truth or report a model as validated.
- **Access date:** 25 October 2026. The dataset's page, count, license, and availability can change.

### Download it manually (no API token in this notebook)

1. Open the linked Roboflow Universe page and review the current licence, attribution, and image contents.
2. Use the site's **Download Dataset** flow if available for your account and choose an export that gives you the original images and class labels. Export format options can change; this notebook deliberately does not invent a direct archive URL or require a secret.
3. Extract the images under `/content/ergo_dataset/images/`.
4. Create `/content/ergo_dataset/annotations.csv` with columns `image_path,label` and optional `group_id`. `image_path` should be relative to `/content/ergo_dataset/` (for example `images/example.jpg`). `label` must reproduce a source label exactly. Add `group_id` only when you have a real subject/recording identifier from the dataset metadata; do not fabricate IDs.
5. Keep downloaded imagery in Colab or a private working folder. Do not commit personal images or large dataset files to GitHub.

**Alternative:** use another dataset only after you verify its source, license, labels, and suitability. Keep its citation and license terms with your experiment notes.

## 3. Dataset location and schema validation

The notebook does not silently download, fabricate, or replace data. Without an image folder and manifest, downstream analysis is skipped and you can still inspect the notebook's workflow. A manifest permits labels and optional subject/recording groups to be explicit.

In [ ]:
DATA_ROOT = Path("/content/ergo_dataset")
IMAGE_ROOT = DATA_ROOT / "images"
MANIFEST_PATH = DATA_ROOT / "annotations.csv"
OUTPUT_DIR = Path("/content/ergovision_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_COLUMNS = [
    "trunkInclinationDeg", "headNeckProxyDeg", "leftElbowAngleDeg",
    "rightElbowAngleDeg", "leftArmTrunkSeparationDeg", "rightArmTrunkSeparationDeg"
]

DATA_READY = False
manifest = pd.DataFrame()

if MANIFEST_PATH.is_file():
    manifest = pd.read_csv(MANIFEST_PATH)
    required_columns = {"image_path", "label"}
    missing_columns = required_columns - set(manifest.columns)
    if missing_columns:
        raise ValueError(f"annotations.csv is missing columns: {sorted(missing_columns)}")
    manifest["image_path"] = manifest["image_path"].astype(str)
    manifest["label"] = manifest["label"].astype(str).str.strip()
    manifest = manifest[manifest["label"].ne("") & manifest["image_path"].ne("")].copy()
    manifest["resolved_path"] = manifest["image_path"].apply(
        lambda value: Path(value) if Path(value).is_absolute() else DATA_ROOT / value
    )
    missing_files = manifest.loc[~manifest["resolved_path"].apply(Path.is_file), "image_path"].tolist()
    if missing_files:
        raise FileNotFoundError(f"Manifest references missing images (first 10): {missing_files[:10]}")
    if manifest["label"].nunique() < 2:
        raise ValueError("At least two distinct classes are required for supervised classification.")
    if len(manifest) < 12:
        warnings.warn("Fewer than 12 images were supplied; split-based metrics may be unstable or impossible.")
    DATA_READY = True
    print(f"Loaded {len(manifest)} labelled image records from {MANIFEST_PATH}")
else:
    print("DATASET NOT INSTALLED. No images, labels, or results have been fabricated.")
    print("Follow the acquisition instructions above, then add:", MANIFEST_PATH)
    print("Required columns: image_path,label; optional column: group_id (real subject/recording ID).")

## 4. Inspect the manifest and labels

Do not continue with labels whose meaning you cannot explain. The notebook shows class counts and a small sample of paths; it does not infer what `Good` or `Bad` means beyond the source-provided label.

In [ ]:
if DATA_READY:
    display(manifest.head(10))
    print("Unique labels:", sorted(manifest["label"].unique().tolist()))
    print("Manifest columns:", manifest.columns.tolist())
    print("Missing values:\n", manifest[["image_path", "label"]].isna().sum())
    ax = manifest["label"].value_counts().sort_index().plot(kind="bar", figsize=(6, 3), title="Source label distribution")
    ax.set_xlabel("Source label"); ax.set_ylabel("Image records"); plt.tight_layout(); plt.show()
else:
    print("Skipped: install and verify the dataset manifest first.")

## 5. Extract the same six pose features used by the web app

MediaPipe Pose Landmarker/Solutions estimates landmarks from an image. The functions below use normalized image `x/y` converted to pixel coordinates for geometry. They never interpret the model's normalized `z` as a measured anatomical depth. The app and notebook intentionally use the same named feature order.

In [ ]:
# MediaPipe's 33-landmark pose schema uses these indices for the relevant body points.
LANDMARKS = {
    "left_ear": 7, "right_ear": 8,
    "left_shoulder": 11, "right_shoulder": 12,
    "left_elbow": 13, "right_elbow": 14,
    "left_wrist": 15, "right_wrist": 16,
    "left_hip": 23, "right_hip": 24,
}
MIN_VISIBILITY = 0.65


def angle_at_point(a, b, c):
    """Internal 2D angle ABC in degrees; None for degenerate geometry."""
    vectors = (np.asarray(a, dtype=float) - np.asarray(b, dtype=float),
               np.asarray(c, dtype=float) - np.asarray(b, dtype=float))
    if not all(np.isfinite(v).all() for v in vectors):
        return None
    norms = [float(np.linalg.norm(v)) for v in vectors]
    if min(norms) < 1e-9:
        return None
    cosine = float(np.dot(vectors[0], vectors[1]) / (norms[0] * norms[1]))
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))


def angle_between_vectors(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    norms = [float(np.linalg.norm(a)), float(np.linalg.norm(b))]
    if not np.isfinite(a).all() or not np.isfinite(b).all() or min(norms) < 1e-9:
        return None
    return float(np.degrees(np.arccos(np.clip(float(np.dot(a, b) / (norms[0] * norms[1])), -1.0, 1.0))))


def extract_pose_features(pose_landmarks, image_width, image_height, min_visibility=MIN_VISIBILITY):
    result = {name: np.nan for name in FEATURE_COLUMNS}
    if pose_landmarks is None or len(pose_landmarks) < 25 or image_width <= 0 or image_height <= 0:
        return result

    def get(name):
        point = pose_landmarks[LANDMARKS[name]]
        visibility = getattr(point, "visibility", None)
        presence = getattr(point, "presence", None)
        if visibility is None or visibility < min_visibility or (presence is not None and presence < min_visibility):
            return None
        if not np.isfinite([point.x, point.y]).all():
            return None
        return np.array([point.x * image_width, point.y * image_height], dtype=float)

    ls, rs = get("left_shoulder"), get("right_shoulder")
    lh, rh = get("left_hip"), get("right_hip")
    le, re = get("left_ear"), get("right_ear")
    if all(point is not None for point in (ls, rs, lh, rh)):
        shoulders, hips = (ls + rs) / 2.0, (lh + rh) / 2.0
        dx, dy = shoulders[0] - hips[0], shoulders[1] - hips[1]
        if np.hypot(dx, dy) >= 1e-9:
            result["trunkInclinationDeg"] = abs(float(np.degrees(np.arctan2(dx, -dy))))
        ears = [point for point in (le, re) if point is not None]
        if ears:
            ear_mid = np.mean(ears, axis=0)
            result["headNeckProxyDeg"] = angle_between_vectors(ear_mid - shoulders, shoulders - hips)

    for side in ("left", "right"):
        shoulder, elbow, wrist, hip = (get(f"{side}_{part}") for part in ("shoulder", "elbow", "wrist", "hip"))
        # For arm/trunk separation, the elbow-to-shoulder vector is compared at the shoulder with shoulder-to-hip.
        if shoulder is not None and elbow is not None and wrist is not None:
            result[f"{side}ElbowAngleDeg"] = angle_at_point(shoulder, elbow, wrist)
        if shoulder is not None and elbow is not None and hip is not None:
            result[f"{side}ArmTrunkSeparationDeg"] = angle_at_point(elbow, shoulder, hip)
    return result


def read_and_extract(path):
    image = cv2.imread(str(path))
    if image is None:
        return None, "image_read_failed"
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    with pose_estimator_lock:
        result = pose_estimator.process(rgb)
    if not result.pose_landmarks:
        return None, "pose_not_detected"
    features = extract_pose_features(result.pose_landmarks.landmark, image.shape[1], image.shape[0])
    essential = [features["trunkInclinationDeg"], features["headNeckProxyDeg"]]
    elbow_available = any(np.isfinite(features[key]) for key in ("leftElbowAngleDeg", "rightElbowAngleDeg"))
    if not all(np.isfinite(value) for value in essential) or not elbow_available:
        return None, "insufficient_landmarks"
    return features, "ok"

print("Feature columns:", FEATURE_COLUMNS)

## 6. Process images and clean invalid observations

Pose processing can fail on cropped, occluded, or low-quality images. Those examples are logged and skipped, never converted into a fake zero-angle sample. This can change the observed class balance, so review the exclusion table.

In [ ]:
feature_frame = pd.DataFrame()
exclusion_frame = pd.DataFrame()

if DATA_READY:
    pose_estimator_lock = __import__("threading").Lock()
    pose_estimator = mp.solutions.pose.Pose(
        static_image_mode=True, model_complexity=1,
        enable_segmentation=False,
        min_detection_confidence=0.5
    )
    records, exclusions = [], []
    try:
        for row in manifest.itertuples(index=False):
            image_path = Path(row.resolved_path)
            extracted, status = read_and_extract(image_path)
            if extracted is None:
                exclusions.append({"image_path": str(row.image_path), "label": row.label, "reason": status})
                continue
            records.append({
                "image_path": str(row.image_path), "label": str(row.label),
                "group_id": getattr(row, "group_id", None), **extracted
            })
    finally:
        pose_estimator.close()
    feature_frame = pd.DataFrame(records)
    exclusion_frame = pd.DataFrame(exclusions, columns=["image_path", "label", "reason"])
    print(f"Extracted features for {len(feature_frame)} images; excluded {len(exclusion_frame)}.")
    if len(exclusion_frame):
        display(exclusion_frame["reason"].value_counts().rename_axis("reason").to_frame("count"))
    if len(feature_frame):
        display(feature_frame.head())
else:
    print("Skipped: waiting for verified images and labels.")

## 7. Feature distributions and class balance

These plots describe the extracted sample only. A relationship between a feature and a source label does not establish medical causation or tell us that a posture causes injury.

In [ ]:
if DATA_READY and len(feature_frame):
    display(feature_frame[FEATURE_COLUMNS].describe().T)
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for ax, column in zip(axes.flat, FEATURE_COLUMNS):
        for label, group in feature_frame.groupby("label"):
            ax.hist(group[column].dropna(), bins=15, alpha=0.48, label=str(label))
        ax.set_title(column, fontsize=9); ax.set_xlabel("Degrees"); ax.set_ylabel("Images")
    axes.flat[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(feature_frame["label"].value_counts().rename_axis("label").to_frame("valid feature samples"))
else:
    print("Skipped: no valid extracted samples are available.")

## 8. Define the learning target and split before fitting preprocessing

The source label is the target. Feature imputation and scaling are placed *inside* each scikit-learn pipeline so they are fitted only on that model's training portion. If a valid `group_id` column contains at least three distinct real subject/recording groups, the split is grouped to reduce near-duplicate leakage. If no group metadata exists, a stratified random split is used with an explicit leakage warning.

In [ ]:
SPLIT_READY = False
if DATA_READY and len(feature_frame) >= 12 and feature_frame["label"].nunique() >= 2:
    model_data = feature_frame.dropna(subset=["label"]).copy()
    X = model_data[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = model_data["label"].astype(str)
    if "group_id" in model_data.columns:
        raw_groups = model_data["group_id"].astype("string").str.strip()
        raw_groups = raw_groups.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        has_groups = bool(raw_groups.notna().all() and raw_groups.nunique(dropna=True) >= 3)
        group_values = raw_groups.astype(str)
    else:
        group_values = pd.Series([""] * len(model_data), index=model_data.index)
        has_groups = False

    all_missing_features = X.columns[X.notna().sum() == 0].tolist()
    if all_missing_features:
        raise ValueError(
            "These features are entirely missing in the extracted dataset: "
            f"{all_missing_features}. Review camera views and source data; do not train a mismatched model."
        )

    indices = np.arange(len(model_data))
    if has_groups:
        gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
        trainval_idx, test_idx = next(gss_test.split(X, y, groups=group_values))
        gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE + 1)
        train_rel, val_rel = next(gss_val.split(X.iloc[trainval_idx], y.iloc[trainval_idx], groups=group_values.iloc[trainval_idx]))
        train_idx, val_idx = trainval_idx[train_rel], trainval_idx[val_rel]
        print("Using group-aware train/validation/test split. Group count:", group_values.nunique())
    else:
        warnings.warn("No suitable subject/recording group metadata found. Stratified image-level splitting may leak near-duplicates across splits; do not claim subject-independent generalisation.")
        trainval_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
        train_idx, val_idx = train_test_split(trainval_idx, test_size=0.25, random_state=RANDOM_STATE + 1, stratify=y.iloc[trainval_idx])

    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
    X_trainval, y_trainval = X.iloc[trainval_idx], y.iloc[trainval_idx]
    split_info = pd.DataFrame({"split": ["train", "validation", "test"], "rows": [len(train_idx), len(val_idx), len(test_idx)]})
    display(split_info)
    display(pd.DataFrame({"train": y_train.value_counts(), "validation": y_val.value_counts(), "test": y_test.value_counts()}).fillna(0).astype(int))
    if y_train.nunique() < 2 or y_val.nunique() < 2 or y_test.nunique() < 2:
        warnings.warn("At least one split contains fewer than two classes. Metrics may be incomplete; revisit sample size and group distribution.")
    SPLIT_READY = True
else:
    print("Training skipped. Need at least 12 usable images with two or more labels; use the EDA and source-label checks first.")

## 9. Baseline and model comparison

The majority-class dummy model is the baseline. Logistic regression is a simple linear classifier; random forest can model nonlinear relationships. Candidate selection uses the validation macro-F1, which gives each class equal weight. The held-out test set remains untouched until the selected model is evaluated once.

In [ ]:
validation_results = pd.DataFrame()
chosen_name = None
chosen_model = None
if SPLIT_READY:
    candidates = {
        "Majority baseline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyClassifier(strategy="most_frequent"))
        ]),
        "Logistic regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
        ]),
        "Random forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=250, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE))
        ])
    }
    rows = []
    for name, candidate in candidates.items():
        candidate.fit(X_train, y_train)
        prediction = candidate.predict(X_val)
        rows.append({
            "model": name,
            "validation_accuracy": accuracy_score(y_val, prediction),
            "validation_macro_precision": precision_score(y_val, prediction, average="macro", zero_division=0),
            "validation_macro_recall": recall_score(y_val, prediction, average="macro", zero_division=0),
            "validation_macro_f1": f1_score(y_val, prediction, average="macro", zero_division=0),
        })
    validation_results = pd.DataFrame(rows).sort_values("validation_macro_f1", ascending=False)
    display(validation_results)
    chosen_name = str(validation_results.iloc[0]["model"])
    chosen_model = candidates[chosen_name]
    print("Selected on validation macro-F1:", chosen_name)
else:
    print("Skipped: no valid train/validation/test split.")

## 10. Final test evaluation

All metrics below are computed only when a valid held-out test set exists. Accuracy, precision, recall and F1 each describe different aspects of classification. With a small dataset, the values can vary considerably across different splits and should not be generalised beyond this experiment.

In [ ]:
test_predictions = None
final_metrics = None
if SPLIT_READY and chosen_model is not None:
    # Refit preprocessing and the selected model on train + validation, then evaluate once on test.
    chosen_model.fit(X_trainval, y_trainval)
    test_predictions = chosen_model.predict(X_test)
    final_metrics = {
        "accuracy": accuracy_score(y_test, test_predictions),
        "macro_precision": precision_score(y_test, test_predictions, average="macro", zero_division=0),
        "macro_recall": recall_score(y_test, test_predictions, average="macro", zero_division=0),
        "macro_f1": f1_score(y_test, test_predictions, average="macro", zero_division=0),
    }
    display(pd.DataFrame([final_metrics]).T.rename(columns={0: "held-out test score"}))
    print(classification_report(y_test, test_predictions, zero_division=0))
    labels = sorted(y.unique())
    ConfusionMatrixDisplay(confusion_matrix(y_test, test_predictions, labels=labels), display_labels=labels).plot(cmap="Blues", values_format="d")
    plt.title("Held-out test confusion matrix"); plt.tight_layout(); plt.show()
else:
    print("Skipped: the test metrics have not been calculated because the dataset/split/model is unavailable.")

## 11. Error analysis and limitations

Inspect individual mistakes using image paths and labels—do not interpret a wrong label as evidence that the person's posture is medically harmful or safe. Keep a record of image-level exclusions, class balance, and whether group splitting was possible.

In [ ]:
if SPLIT_READY and test_predictions is not None:
    error_analysis = model_data.iloc[test_idx][["image_path", "label"]].copy()
    error_analysis["predicted_label"] = test_predictions
    error_analysis["correct"] = error_analysis["label"].astype(str) == error_analysis["predicted_label"].astype(str)
    display(error_analysis.loc[~error_analysis["correct"]])
    print("Test rows:", len(error_analysis), "| errors:", int((~error_analysis["correct"]).sum()))
else:
    print("Skipped: no held-out predictions to inspect.")

## 12. Save the trained model and a metadata sidecar

The notebook saves the selected **feature classifier pipeline** as a local joblib artifact. It does not save or train the MediaPipe pose model. A joblib file is for Python use; it cannot be loaded directly in a browser. The web app will continue using its transparent rule-based screening unless a compatible optional inference adapter is explicitly implemented and tested.

In [ ]:
MODEL_PATH = OUTPUT_DIR / "ergovision_classifier.joblib"
METADATA_PATH = OUTPUT_DIR / "ergovision_classifier_metadata.json"
if SPLIT_READY and chosen_model is not None and final_metrics is not None:
    joblib.dump(chosen_model, MODEL_PATH)
    metadata = {
        "project": "ErgoVision AI",
        "artifact_type": "optional supervised feature classifier",
        "not_a_medical_device": True,
        "feature_order": FEATURE_COLUMNS,
        "target_labels": sorted(y.unique().tolist()),
        "selected_model": chosen_name,
        "validation_macro_f1": float(validation_results.iloc[0]["validation_macro_f1"]),
        "held_out_test_metrics": {key: float(value) for key, value in final_metrics.items()},
        "dataset_source": "Roboflow Universe sitting posture dataset by IKORNProject, or user-verified compatible source",
        "license_review_required": True,
        "group_aware_split_used": bool(has_groups),
        "note": "Metrics are experimental and not evidence of clinical validity. Inspect label semantics and dataset license."
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print("Saved model:", MODEL_PATH)
    print("Saved metadata:", METADATA_PATH)
    display(pd.read_json(METADATA_PATH, typ="series").to_frame("value"))
else:
    print("No model artifact written. Training/evaluation has not produced a valid result yet.")

## 13. Optional artifact download

Download only after reviewing the model's label quality, error analysis, and held-out test results. Keep the artifact out of the Git repository unless you have a reason and permission to distribute it.

In [ ]:
if SPLIT_READY and chosen_model is not None and final_metrics is not None:
    try:
        from google.colab import files
        files.download(str(MODEL_PATH))
        files.download(str(METADATA_PATH))
    except Exception as exc:
        print("Automatic download is available in Google Colab; otherwise retrieve the files from:", OUTPUT_DIR)
        print("Download UI unavailable:", type(exc).__name__)
else:
    print("Skipped: no trained artifact is available.")

## 14. Optional integration with ErgoVision AI

1. **Pose model:** the website already uses MediaPipe's pretrained model and does not depend on this notebook.
2. **Feature schema:** the browser's `toMlFeatureVector()` in `frontend/src/utils/mlFeatures.ts` uses the six columns listed above in the same order. Every feature must be present; the app intentionally does not invent or impute missing live values.
3. **Python experiment:** from the repository root, `ml/classifier.py` checks the feature count/order when model metadata is available and predicts using the fixed feature contract. Use this only with a model from a verified training run.
4. **Browser deployment:** `joblib` is Python-specific. To perform inference in the browser, a maintainer would need to export the trained pipeline to an interoperable format such as ONNX, add a compatible browser runtime, map input/output names, and test that browser predictions match Python predictions on a fixed set. This repository does **not** include a trained artifact or an ONNX runtime adapter, so the UI must continue to show rule-based screening only until that work is done.
5. **Model limits:** even a strong test metric on this small label set does not establish clinical validity, causal relationships, or performance on new people and cameras.

## 15. Conclusion template (complete only after running)

Do not pre-fill claims. After you execute this notebook with a reviewed dataset, write down:

- The exact dataset source/version and license terms you checked.
- How many records were downloaded, excluded for missing pose, and used for training/testing.
- Whether a real subject/recording group split was possible.
- The validation-selected model and the actual held-out accuracy, macro-precision, macro-recall, macro-F1, and confusion matrix.
- The largest error patterns you observed and why the labels/dataset may not generalise.
- A conclusion phrased as an exploratory result, not a diagnosis, injury predictor, or official RULA result.

**No experimental metrics are included in this blank notebook.** They appear only after successful execution with the dataset that you supply.